In [ ]:
import pandas as pd
from sklearn.metrics import mean_absolute_error
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold
from sklearn.linear_model import LinearRegression

In [ ]:
data_1 = pd.read_csv('/datasets/gold_recovery_train.csv')
data_2 = pd.read_csv('/datasets/gold_recovery_test.csv')
data_3 = pd.read_csv('/datasets/gold_recovery_full.csv')

In [ ]:
data_1.info()

In [ ]:
data_2.info()

In [ ]:
data_3.info()

In [ ]:
print('VALORES NULOS DEL PRIMER DATAFRAME')
print(data_1.isna().sum().sort_values(ascending=False))
print()
print('VALORES NULOS DEL SEGUNDO DATAFRAME')
print(data_2.isna().sum().sort_values(ascending=False))
print()
print('VALORES NULOS DEL TERCER DATAFRAME')
print(data_3.isna().sum().sort_values(ascending=False))

In [ ]:
data_1.columns.tolist()

In [ ]:
[col for col in data_1.columns if 'rougher' in col and 'au' in col]

In [ ]:
numerador = data_1['rougher.output.concentrate_au'] * (data_1['rougher.input.feed_au'] - data_1['rougher.output.tail_au'])
denominador = data_1['rougher.input.feed_au'] * (data_1['rougher.output.concentrate_au'] - data_1['rougher.output.tail_au'])
recuperacion = (numerador / denominador) * 100
mascara = (recuperacion.notna()) & (data_1['rougher.output.recovery'].notna())
recuperacion_filtrada = recuperacion[mascara]
recovery_filtrado = data_1['rougher.output.recovery'][mascara]
EAM = mean_absolute_error(recuperacion_filtrada, recovery_filtrado)

In [ ]:
#Esto lo puse por mi propia claridad, no me dejaba calcular el EAM si cada columna tenia valores nulos
print('La variable de recuperacion tiene:', recuperacion.isna().sum(), 'valores nulos')
print('La variable de recovery tiene:', data_1['rougher.output.recovery'].isna().sum(), 'valores nulos')

In [ ]:
print(f'La diferencia entre la columna del data_1 y mi calculo es: {EAM:.15f}')

In [ ]:
df_train = set(data_1.columns)
df_test = set(data_2.columns)
diferencia = df_train - df_test
print(diferencia)
print('La cantidad de columnas faltantes es:', len(diferencia))

In [ ]:
diferencia = list(diferencia)
data_faltantes = data_1[diferencia]
data_faltantes.dtypes

### Análisis de columnas ausentes en el conjunto de prueba (`data_2`)

Al comparar el conjunto de prueba (`data_2`) con el conjunto de datos completo (`data_1`), se identificaron las siguientes características en las características ausentes:

* **Cantidad de columnas faltantes:** 34 columnas.
* **Tipo de variables:** Todas las columnas correspondientes son de tipo numérico continuo (`float64`).
* **Naturaleza de los parámetros:** Pertenecen a las categorías de **salida** (`output`) y **cálculos derivados** (`calculation`).
* **Justificación técnica:** Estos parámetros corresponden a mediciones tomadas o calculadas en etapas posteriores al proceso tecnológico. En un entorno de producción real, no están disponibles al momento de realizar la predicción, por lo que se excluyen del conjunto de prueba para evitar el sesgo de anticipación (*data leakage*).

In [ ]:
data_1['date'] = pd.to_datetime(data_1['date'])
data_2['date'] = pd.to_datetime(data_2['date'])
data_3['date'] = pd.to_datetime(data_3['date'])
print(data_1.dtypes)
print(data_2.dtypes)
print(data_3.dtypes)

In [ ]:

print(data_1['date'].is_monotonic_increasing)
print(data_2['date'].is_monotonic_increasing)
print(data_3['date'].is_monotonic_increasing)
data_1 = data_1.set_index('date')
data_2 = data_2.set_index('date')
data_3 = data_3.set_index('date')
print(data_1.index)
print(data_2.index)
print(data_3.index)


In [ ]:
targets = ['rougher.output.recovery', 'final.output.recovery']

targets_data_1 = data_1[targets].copy()
targets_data_3 = data_3[targets].copy()

data_1 = data_1.ffill()
data_2 = data_2.ffill()
data_3 = data_3.ffill()

data_1[targets] = targets_data_1
data_3[targets] = targets_data_3

print("Nulos en targets de data_1 (train):\n", data_1[targets].isna().sum())
print("\nNulos en targets de data_3 (full):\n", data_3[targets].isna().sum())

### Resumen del preprocesamiento de datos

Se llevó a cabo la preparación y limpieza de los tres conjuntos de datos (`data_1`, `data_2` y `data_3`) mediante los siguientes pasos:

1. **Conversión de formato:** Se transformó la columna `date` de tipo `object` a formato `datetime` en todos los DataFrames.
2. **Validación de secuencia temporal:** Se verificó que los registros estuvieran ordenados cronológicamente de forma ascendente.
3. **Asignación de índice:** Se estableció la columna `date` como el índice (`index`) de cada DataFrame para facilitar el análisis de series de tiempo.
4. **Tratamiento de valores nulos:** Se aplicó el método de imputación `ffill()` (propogación hacia adelante) para rellenar los datos faltantes con el último valor válido anterior, respetando la continuidad temporal del proceso.
5. **Verificación final:** Se reevaluó la presencia de valores nulos en todas las columnas, confirmando una cuenta de 0 faltantes tras la imputación.

In [ ]:
metales = ["au", "ag", "pb"]
for met in metales:
    resultado = [col for col in data_3.columns if met in col and 'calculation' not in col]
    print(met, resultado)

In [ ]:
au_1 = data_3['rougher.output.concentrate_au'].mean()
au_2 = data_3['primary_cleaner.output.concentrate_au'].mean()
au_3 = data_3['final.output.concentrate_au'].mean()
print(au_1)
print(au_2)
print(au_3)
print('--------------------')
ag_1 = data_3['rougher.output.concentrate_ag'].mean()
ag_2 = data_3['primary_cleaner.output.concentrate_ag'].mean()
ag_3 = data_3['final.output.concentrate_ag'].mean()
print(ag_1)
print(ag_2)
print(ag_3)
print('--------------------')
pb_1 = data_3['rougher.output.concentrate_pb'].mean()
pb_2 = data_3['primary_cleaner.output.concentrate_pb'].mean()
pb_3 = data_3['final.output.concentrate_pb'].mean()
print(pb_1)
print(pb_2)
print(pb_3)

In [ ]:
data_1.columns[data_1.columns.str.contains('feed_size')]

In [ ]:
plt.figure(figsize=(10, 6))

data_1['rougher.input.feed_size'].dropna().hist(
    bins=50, 
    density=True, 
    alpha=0.5, 
    label='Train', 
    color='tab:blue'
)

data_2['rougher.input.feed_size'].dropna().hist(
    bins=50, 
    density=True, 
    alpha=0.5, 
    label='Test', 
    color='tab:orange'
)

plt.title('Distribución del tamaño de partícula de alimentación (rougher.input.feed_size)', fontsize=13, fontweight='bold')
plt.xlabel('Tamaño de partícula (feed_size)', fontsize=11)
plt.ylabel('Densidad de frecuencia', fontsize=11)
plt.xlim(0, 150) 
plt.legend(title='Dataset', frameon=True)
plt.grid(True, linestyle='--', alpha=0.6)

plt.show()

### Análisis de la distribución del tamaño de partícula (`feed_size`)

Tras comparar los histogramas y la densidad de distribución del tamaño de partícula de alimentación (`rougher.input.feed_size`) entre los conjuntos de datos de entrenamiento (`train`) y prueba (`test`):

* **Similitud observada:** Las distribuciones presentan una forma y rango altamente consistentes entre ambos conjuntos.
* **Impacto en el modelado:** La homogeneidad en `feed_size` confirma que el conjunto de prueba es representativo de las condiciones del proceso real, garantizando que el modelo pueda generalizar adecuadamente sin sufrir sesgos por diferencias en la alimentación.

In [ ]:
raw_feed_train = (data_1['rougher.input.feed_ag'] + 
                  data_1['rougher.input.feed_pb'] + 
                  data_1['rougher.input.feed_sol'] + 
                  data_1['rougher.input.feed_au'])

rougher_conc_train = (data_1['rougher.output.concentrate_ag'] + 
                      data_1['rougher.output.concentrate_pb'] + 
                      data_1['rougher.output.concentrate_sol'] + 
                      data_1['rougher.output.concentrate_au'])

final_conc_train = (data_1['final.output.concentrate_ag'] + 
                    data_1['final.output.concentrate_pb'] + 
                    data_1['final.output.concentrate_sol'] + 
                    data_1['final.output.concentrate_au'])

valid_indices = data_1[
    (raw_feed_train > 0) & 
    (rougher_conc_train > 0) & 
    (final_conc_train > 0)
].index


raw_feed_full = (data_3['rougher.input.feed_ag'] + 
                 data_3['rougher.input.feed_pb'] + 
                 data_3['rougher.input.feed_sol'] + 
                 data_3['rougher.input.feed_au'])

rougher_conc_full = (data_3['rougher.output.concentrate_ag'] + 
                     data_3['rougher.output.concentrate_pb'] + 
                     data_3['rougher.output.concentrate_sol'] + 
                     data_3['rougher.output.concentrate_au'])

final_conc_full = (data_3['final.output.concentrate_ag'] + 
                   data_3['final.output.concentrate_pb'] + 
                   data_3['final.output.concentrate_sol'] + 
                   data_3['final.output.concentrate_au'])

valid_indices_full = data_3[
    (raw_feed_full > 0) & 
    (rougher_conc_full > 0) & 
    (final_conc_full > 0)
].index

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))
plt.hist(raw_feed_full, bins=50, alpha=0.5, label='Materia Prima (Raw Feed)')
plt.hist(rougher_conc_full, bins=50, alpha=0.5, label='Concentrado Rougher')
plt.hist(final_conc_full, bins=50, alpha=0.5, label='Concentrado Final')

plt.title('Distribución de Concentraciones Totales por Etapa')
plt.xlabel('Concentración Total (%)')
plt.ylabel('Frecuencia')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
concentraciones_df = pd.DataFrame({
    'Materia Prima (Raw Feed)': raw_feed_full,
    'Concentrado Rougher': rougher_conc_full,
    'Concentrado Final': final_conc_full
})

# Ver métricas clave (mínimos, promedios, etc.)
concentraciones_df.describe()

In [ ]:
ceros_raw = (raw_feed_full == 0).sum()
ceros_rougher = (rougher_conc_full == 0).sum()
ceros_final = (final_conc_full == 0).sum()

print(f"Valores en cero en Materia Prima (Raw Feed): {ceros_raw}")
print(f"Valores en cero en Concentrado Rougher: {ceros_rougher}")
print(f"Valores en cero en Concentrado Final: {ceros_final}")

In [ ]:
print("Filas totales en data_3:", len(data_3))
print("Filas válidas (sin anomalías) en data_3:", len(valid_indices_full))
print("Dimensiones de data_3 filtrado:", data_3.loc[valid_indices_full].shape)

### Conclusión: Detección y eliminación de anomalías

Tras analizar las concentraciones totales en las tres etapas clave del proceso (materia prima `rougher.input.feed_*`, concentrado rougher y concentrado final), se identificaron valores anómalos con concentraciones iguales a cero. 

* **Filas eliminadas:** Se removieron un total de **2,296 filas** (pasando de 22,716 a 20,420 observaciones válidas en el conjunto completo `data_3`).
* **Justificación:** Una concentración total de 0% en cualquier etapa no representa un escenario real ni tiene sentido físico en el procesamiento de mineral, por lo que estas observaciones corresponden a errores de medición o datos faltantes que distorsionarían el aprendizaje de los modelos.

Al aplicar la máscara de índices `valid_indices_full`, los datos quedan limpios y listos para pasar a la fase de modelado sin necesidad de duplicar datasets en memoria.

In [ ]:
def smape(y_true, y_pred):
    """
    Calcula el sMAPE aplicando el valor absoluto a cada término por separado en el denominador.
    """
    numerator = np.abs(y_true - y_pred)
    
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2
    
    smape_val = np.where(denominator == 0, 0, numerator / denominator)
    return np.mean(smape_val) * 100

def final_smape(y_true, y_pred):
    """
    Calcula el sMAPE final ponderado: 25% rougher y 75% final.
    """
    smape_rougher = smape(y_true.iloc[:, 0], y_pred[:, 0])
    smape_final = smape(y_true.iloc[:, 1], y_pred[:, 1])
    
    return 0.25 * smape_rougher + 0.75 * smape_final

In [ ]:
# Definir las características (features) excluyendo 'date'
features = [col for col in data_2.columns if col != 'date']

# --- ENTRENAMIENTO (Train) ---
# 1. Filas sin nulos en targets de train (data_1)
non_null_target_indices_train = data_1.dropna(subset=['rougher.output.recovery', 'final.output.recovery']).index

# 2. Intersección de índices para train: sin nulos en target Y sin anomalías
train_indices = non_null_target_indices_train.intersection(valid_indices)

x_train = data_1.loc[train_indices, features]
y_train = data_1.loc[train_indices, ['rougher.output.recovery', 'final.output.recovery']]


# --- PRUEBA (Test) ---
# 1. Filas sin nulos en targets de full (data_3)
non_null_target_indices_test = data_3.dropna(subset=['rougher.output.recovery', 'final.output.recovery']).index

# 2. Intersección para test: usa valid_indices_full y data_3
test_indices = data_2.index.intersection(valid_indices_full).intersection(non_null_target_indices_test)

x_test = data_2.loc[test_indices, features]
y_test = data_3.loc[test_indices, ['rougher.output.recovery', 'final.output.recovery']]

# Verificación de dimensiones
print(f"Dimensiones de x_train: {x_train.shape}, y_train: {y_train.shape}")
print(f"Dimensiones de x_test: {x_test.shape}, y_test: {y_test.shape}")

In [ ]:
param_grid = {
    'max_depth': [2, 4, 6, 8, 10],
    'n_estimators': [10, 20]
}

best_smape = float('inf')
best_params = {}

kf = KFold(n_splits=5, shuffle=True, random_state=12345)

for depth in param_grid['max_depth']:
    for est in param_grid['n_estimators']:
        scores_fold = []
        
        for train_index, valid_index in kf.split(x_train):
            x_train_fold = x_train.iloc[train_index]
            y_train_fold = y_train.iloc[train_index]
            x_valid_fold = x_train.iloc[valid_index]
            y_valid_fold = y_train.iloc[valid_index]

            model = RandomForestRegressor(n_estimators=est, max_depth=depth, random_state=12345)
            model.fit(x_train_fold, y_train_fold)
            
            prediccion_forest = model.predict(x_valid_fold)
            # CORRECCIÓN: Se cambia 'metrica' por 'final_smape'
            score = final_smape(y_valid_fold, prediccion_forest)
            scores_fold.append(score)

        smape_promedio = np.mean(scores_fold)
        print(f"max_depth={depth}, n_estimators={est} -> sMAPE promedio: {smape_promedio:.4f}")

        if smape_promedio < best_smape:
            best_smape = smape_promedio
            best_params = {'max_depth': depth, 'n_estimators': est}

print("\n" + "="*40)
print(f"¡Mejores hiperparámetros encontrados!")
print(f"Parámetros: {best_params}")
print(f"Mejor sMAPE promedio: {best_smape:.4f}")

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=12345)

resultados_forest = []

for train_index, valid_index in kf.split(x_train):
    x_train_fold = x_train.iloc[train_index]
    y_train_fold = y_train.iloc[train_index]
    x_valid_fold = x_train.iloc[valid_index]
    y_valid_fold = y_train.iloc[valid_index]

    model = RandomForestRegressor(n_estimators=10, max_depth=10, random_state=12345)
    model.fit(x_train_fold, y_train_fold)
    
    prediccion_forest = model.predict(x_valid_fold)
    
    # CAMBIO AQUÍ: 'metrica' se reemplaza por 'final_smape'
    resultado = final_smape(y_valid_fold, prediccion_forest)
    resultados_forest.append(resultado)

print(resultados_forest)
print('El promedio es:', np.mean(resultados_forest))

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=12345)

resultados_lineal = []

for train_index, valid_index in kf.split(x_train):
    x_train_fold = x_train.iloc[train_index]
    y_train_fold = y_train.iloc[train_index]
    x_valid_fold = x_train.iloc[valid_index]
    y_valid_fold = y_train.iloc[valid_index]

    model = LinearRegression()
    model.fit(x_train_fold, y_train_fold)
    
    prediccion_lineal = model.predict(x_valid_fold)
    
    # CAMBIO AQUÍ: 'metrica' se reemplaza por 'final_smape'
    resultado_lineal = final_smape(y_valid_fold, prediccion_lineal)
    resultados_lineal.append(resultado_lineal)

print(resultados_lineal)
print('El promedio es:', np.mean(resultados_lineal))

In [ ]:
model = RandomForestRegressor(n_estimators=10, max_depth=10, random_state=12345)
model.fit(x_train, y_train)

prediccion_forest = model.predict(x_test)

# CAMBIO AQUÍ: 'metrica' se reemplaza por 'final_smape'
resultado_final = final_smape(y_test, prediccion_forest)
print("sMAPE final en test:", resultado_final)

### Conclusión General del Proyecto

#### 1. Preparación y Limpieza de Datos
* **Verificación del cálculo de recuperación:** Se calculó la efectividad de la recuperación de la sustancia principal (`rougher.output.recovery`) y se comparó contra el valor registrado en el dataset, obteniendo un Error Absoluto Medio (MAE) de 9.44e-15, lo que confirmó la integridad matemática de las mediciones iniciales.
* **Tratamiento de nulos:** Se trataron los valores ausentes mediante técnicas de imputación basadas en la continuidad temporal del proceso industrial (`ffill`), resguardando los valores originales en los objetivos (*targets*) sin imputar datos en ellos.
* **Alineación de características:** Se eliminaron del entrenamiento las variables no presentes en el conjunto de prueba por medirse de manera tardía en el proceso industrial.

#### 2. Análisis Exploratorio de Datos (EDA)
* **Concentración de metales (Au, Ag, Pb):** Se analizó la evolución de la concentración de metales a lo largo de las etapas del proceso, observándose un incremento constante y significativo en la concentración de oro (Au), lo cual valida la efectividad de la purificación.
* **Distribución de partículas:** Las distribuciones del tamaño de partícula (`feed_size`) en los conjuntos de entrenamiento y prueba mostraron un comportamiento similar, asegurando la consistencia para el modelado.
* **Filtro de anomalías:** Se identificaron y eliminaron las observaciones con concentraciones totales anómalas iguales a cero en las tres etapas principales: materia prima (`rougher.input.feed_*`), concentrado rougher y concentrado final.

#### 3. Modelado y Evaluación Final
Se evaluaron los modelos mediante validación cruzada de 5 pliegues (KFold) utilizando la métrica de error ponderado sMAPE corregida:

* **Regresión Lineal (LinearRegression):** Obtuvo un sMAPE promedio ponderado en validación cruzada de **7.29%**.
* **Bosque Aleatorio (RandomForestRegressor):** Logró un sMAPE promedio ponderado en validación cruzada de **5.73%** (configurado con `n_estimators=10` y `max_depth=10`), superando a la Regresión Lineal por un 1.56% absoluto.

#### Evaluación en el Conjunto de Prueba
Al evaluar el modelo seleccionado (**Bosque Aleatorio**) sobre el conjunto de prueba (`x_test` y `y_test`), el resultado obtenido fue un **sMAPE final de 7.39%**.

Obtener un desempeño cercano y alineado entre la validación cruzada (5.73%) y el conjunto de prueba (7.39%) resulta consistente con el comportamiento esperado durante el entrenamiento, lo que sugiere que el modelo mantiene un nivel adecuado de generalización sobre datos no vistos sin mostrar indicios evidentes de sobreajuste (*overfitting*).

#### Recomendación Final
Se recomienda utilizar el modelo **Bosque Aleatorio (RandomForestRegressor)** configurado con `max_depth=10` y `n_estimators=10`, ya que obtuvo la métrica de error más baja en la validación cruzada y demostró una alta estabilidad en la evaluación sobre el conjunto de prueba.